# FlowState r1.1 Benchmarking

| | |
|---|---|
| **Organization** | IBM Research |
| **Version** | FlowState R1 (August 2025) |
| **Parameters** | ~9M |
| **Input Features** | Target series only |

## Key Characteristics

FlowState introduces a timescale-invariant forecasting approach through a two-component architecture. An SSM-based encoder maps input time series into a continuous latent space that is independent of the original sampling rate. The Functional Basis Decoder (FBD) then generates forecasts by projecting this representation onto a set of continuous basis functions, allowing adaptation to any target sampling rate at inference time via a scale factor parameter without retraining.

At time of release, FlowState ranked top 10 on the GIFT-Eval leaderboard for zero-shot models despite having more than 10 times fewer parameters than similar models. All reported results are zero-shot with no overlap between training and evaluation data.

---

## 1. Setup & Data Loading

**Validation split:** Train on periods 1–36, validate on 37–42 (identical to Notebook 6)  


**Target:** `Revenue cons. (anon)` at subsegment level  


**Runtime:** T4 GPU (Google Colab) — set manually: Runtime → Change runtime type → T4 GPU


In [11]:
!pip install -q git+https://github.com/ibm-granite/granite-tsfm.git@gift-flowstate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [12]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

GPU available: True
Device: Tesla T4


In [13]:
# ── Constants ──
PERIOD_COL = 'Anon Period'
TARGET     = 'Revenue cons. (anon)'
SUBSEG_COL = 'TGL Business Subsegment'
VAL_CUTOFF = 36
HORIZON    = 6

# ── Paths ──
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/Coding/data/features/training_subsegment_fs.parquet'

# ── Load ──
full_train = pd.read_parquet(DATA_PATH)
train_df   = full_train[full_train[PERIOD_COL] <= VAL_CUTOFF].copy()
val_df     = full_train[full_train[PERIOD_COL] >  VAL_CUTOFF].copy()

print(f'Full train: {full_train.shape}')
print(f'Train (1–36): {train_df.shape} | Val (37–42): {val_df.shape}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full train: (4237, 110)
Train (1–36): (3524, 110) | Val (37–42): (713, 110)


In [ ]:
# FlowState does not support covariates — only the target series is used as input.                        
# Therefore Feature columns are not loaded. 

# ── Subsegments ──
subsegments = sorted(
    train_df.dropna(subset=[TARGET])
    .groupby(SUBSEG_COL, observed=True)
    .size()
    .pipe(lambda s: s[s >= 2].index)
    .tolist()
)
print(f'Subsegments: {len(subsegments)}')

Subsegments: 117


In [ ]:
# ── Evaluation function ──
def evaluate(results: dict, label: str) -> dict:
    all_true, all_pred = [], []
    for seg, preds in results.items():
        actuals = (val_df[val_df[SUBSEG_COL] == seg]
                   .sort_values(PERIOD_COL)
                   .dropna(subset=[TARGET])[TARGET].values)
        n = min(len(actuals), len(preds))
        all_true.extend(actuals[:n])
        all_pred.extend(preds[:n])
    y, yh = np.array(all_true), np.array(all_pred)
    rmse  = np.sqrt(mean_squared_error(y, yh))
    wmape = np.sum(np.abs(y - yh)) / np.sum(np.abs(y)) * 100
    r2    = r2_score(y, yh)
    print(f'{label} — RMSE: {rmse:,.0f} | wMAPE: {wmape:.1f}% | R²: {r2:.4f}')
    return {'Model': label, 'RMSE': rmse, 'wMAPE': wmape, 'R2': r2}

# ── Results tracker ──
all_metrics = []

---
18.5M-parameter SSM encoder + Functional Basis Decoder. Timescale-invariant zero-shot forecasting.  
Zero-shot — no fine-tuning.

- `scale_factor=2.0` for monthly data (formula: Base_Seasonality 24 / observed seasonality 12 = 2.0)
- Output shape: `(batch, 9 quantiles, horizon, channels)` — index 4 = median (0.5 quantile)

In [16]:
from tsfm_public import FlowStateForPrediction

device = 'cuda' if torch.cuda.is_available() else 'cpu'

flowstate_model = FlowStateForPrediction.from_pretrained(
    'ibm-research/flowstate',
    revision='r1.1',
).to(device).eval()
print(f'FlowState r1.1 loaded on {device}.')

FlowState r1.1 loaded on cuda.


In [17]:
flowstate_results, flowstate_errors = {}, []

SCALE_FACTOR = 2.0  # monthly data: 24 / 12 = 2.0

for i, seg in enumerate(subsegments):
    train_s = (train_df[train_df[SUBSEG_COL] == seg]
               .sort_values(PERIOD_COL)
               .dropna(subset=[TARGET]))
    val_s = (val_df[val_df[SUBSEG_COL] == seg]
             .sort_values(PERIOD_COL)
             .dropna(subset=[TARGET]))
    if len(train_s) < 4 or len(val_s) == 0:
        continue
    try:
        series = train_s[TARGET].values.astype(float)
        horizon = len(val_s)

        # Shape: (context, batch=1, channels=1) — batch_first=False
        series_tensor = (torch.tensor(series, dtype=torch.float32)
                         .unsqueeze(1).unsqueeze(-1).to(device))

        with torch.no_grad():
            forecast = flowstate_model(
                series_tensor,
                scale_factor=SCALE_FACTOR,
                prediction_length=horizon,
                batch_first=False,
            )

        # Output: (batch=1, quantiles=9, horizon, channels=1) — index 4 = median
        preds = forecast.prediction_outputs[0, 4, :, 0].cpu().numpy()
        flowstate_results[seg] = preds

    except Exception as e:
        flowstate_errors.append((seg, str(e)))

    if (i + 1) % 25 == 0:
        print(f'{i+1}/{len(subsegments)}')

print(f'Done. Results: {len(flowstate_results)} | Errors: {len(flowstate_errors)}')
if flowstate_errors:
    for seg, err in flowstate_errors[:3]:
        print(f'  {seg}: {err}')

25/117
50/117
75/117
100/117
Done. Results: 107 | Errors: 0


In [18]:
metrics_flowstate = evaluate(flowstate_results, 'FlowState-r1.1')
all_metrics.append(metrics_flowstate)

FlowState-r1.1 — RMSE: 7,703,018 | wMAPE: 9.3% | R²: 0.9873


In [19]:
# ── Export (uncomment to save) ──
# rows = [{'Subsegment': seg, 'Period': t + 37, 'FlowState_pred': p}
#         for seg, preds in flowstate_results.items()
#         for t, p in enumerate(preds)]
# pd.DataFrame(rows).to_csv('/content/drive/MyDrive/Coding/data/predictions/flowstate_val_results.csv', index=False)
# print('Saved.')